# Prepare Data

In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("/kaggle/input/notebooks/davidvista/pinyin-eval-dataset-labelling/eval-chinese-short-sentences-pinyin.csv")

In [3]:
dataset.head()

,text,pinyin
0,他赢了布什说我很高兴是他赢了,ta1 ying2 le bu4 shen2 shuo1 wo3 hen3 gao1 xin...
1,你根本想不到有多畅销赛义德说,ni3 gen1 ben3 xiang3 bu4 dao4 you3 duo1 chang4...
2,国家主席是个虚职实权掌握在共产党总书记的手里,guo2 jia1 zhu3 xi2 shi4 ge4 xu1 zhi2 shi2 quan...
3,如果你批评以色列政府有些人会因此说你是反犹主义者,ru2 guo3 ni3 pi1 ping2 yi3 se4 lie4 zheng4 fu3...
4,我是一十六号下次请还要叫我啊,wo3 shi4 yi1 shi2 liu4 hao4 xia4 ci4 qing3 hai...


# Bigram Count

## Raw

Raw bigram count (possibly with non-lexical units):

In [4]:
!pip install pypinyin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 11.6 MB/s eta 0:00:00


In [5]:
from collections import Counter
from tqdm import tqdm

bigram_freq = Counter()

for text in tqdm(dataset["text"]):

    text = str(text)

    for i in range(len(text) - 1):

        bigram = text[i:i+2]

        # skip punctuation etc.
        if len(bigram.strip()) != 2:
            continue

        bigram_freq[bigram] += 1

100%|██████████| 181302/181302 [00:04<00:00, 38027.40it/s]


In [6]:
import pandas as pd

bigram_df = pd.DataFrame(
    [
        (word, freq)
        for word, freq in bigram_freq.items()
    ],
    columns=["word", "freq"]
)

bigram_df = bigram_df.sort_values(
    "freq",
    ascending=False
)

bigram_df.head(20)

,word,freq
216,中国,22714
787,一个,15256
342,他们,15143
97,我们,14118
266,美国,12095
1781,可能,9188
347,自己,9090
2457,没有,8317
1081,的一,7933
70,一十,7838


In [7]:
from pypinyin import lazy_pinyin, Style

def get_pinyin(word):
    return " ".join(
        lazy_pinyin(
            word,
            style=Style.TONE3
        )
    )

bigram_df["pinyin"] = bigram_df["word"].apply(
    get_pinyin
)

In [8]:
MIN_FREQ = 100

bigram_df = bigram_df[
    bigram_df["freq"] >= MIN_FREQ
].copy()

len(bigram_df)

7158

In [9]:
bigram_df.tail()

,word,freq,pinyin
9312,决方,100,jue2 fang1
63413,率的,100,lv4 de
30822,列出,100,lie4 chu1
14953,了日,100,le ri4
15685,续的,100,xu4 de


## Word

Count of bigram lexical units (with `jieba`):

In [10]:
!pip install jieba

In [11]:
import jieba
from collections import Counter
from tqdm import tqdm

word_freq = Counter()

for text in tqdm(dataset["text"]):

    words = jieba.lcut(str(text))

    for word in words:

        if len(word) < 2:
            continue

        word_freq[word] += 1

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
  0%|          | 0/181302 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.768 seconds.
Prefix dict has been built successfully.
100%|██████████| 181302/181302 [00:23<00:00, 7761.29it/s]


In [12]:
import pandas as pd

word_df = pd.DataFrame(
    [
        (word, freq)
        for word, freq in word_freq.items()
    ],
    columns=["word", "freq"]
)

word_df = word_df.sort_values(
    "freq",
    ascending=False
)

word_df.head(20)

,word,freq
71,中国,20800
116,他们,15143
281,一个,14404
28,我们,14117
89,美国,11073
118,自己,8939
611,可能,8367
848,没有,8201
1018,这个,6950
376,这些,6681


In [13]:
word_df = word_df[
    word_df["word"].str.len() == 2
].copy()

len(word_df)

56323

In [14]:
word_df.head(20)

,word,freq
71,中国,20800
116,他们,15143
281,一个,14404
28,我们,14117
89,美国,11073
118,自己,8939
611,可能,8367
848,没有,8201
1018,这个,6950
376,这些,6681


In [15]:
from pypinyin import lazy_pinyin, Style

def get_pinyin(word):

    return " ".join(
        lazy_pinyin(
            word,
            style=Style.TONE3
        )
    )

word_df["pinyin"] = word_df["word"].apply(
    get_pinyin
)

In [16]:
MIN_FREQ = 100

selected_word_df = word_df[
    word_df["freq"] >= MIN_FREQ
].copy()

len(selected_word_df)

2673

In [17]:
selected_word_df.tail()

,word,freq,pinyin
6245,买下,100,mai3 xia4
3241,患病,100,huan4 bing4
408,拥挤,100,yong1 ji3
26309,倡导,100,chang4 dao3
5831,特权,100,te4 quan2


# Perturbation

## Prepare Homophones

In [18]:
import pickle as pkl

with open("/kaggle/input/notebooks/davidvista/exact-homophones/homophone_inventory.pkl", "rb") as f:
    homophone_inventory = pkl.load(f)

In [19]:
from collections import defaultdict

char_homophones = {}

for py, data in homophone_inventory.items():

    chars = data["chars"]
    freqs = data["freqs"]

    for i, char in enumerate(chars):

        alternatives = []

        for j, alt in enumerate(chars):

            if i == j:
                continue

            alternatives.append(
                (alt, freqs[j])
            )

        char_homophones[char] = alternatives

## Generation

In [20]:
import random
import numpy as np

random.seed(42)

def perturb_word(word):

    candidates = []

    for idx, char in enumerate(word):

        if char in char_homophones:

            candidates.append(
                (idx, char)
            )

    if not candidates:
        return None

    idx, original_char = random.choice(candidates)

    replacements = char_homophones[original_char]

    replacement_chars = [
        c for c, _ in replacements
    ]

    replacement_freqs = np.array(
        [f for _, f in replacements],
        dtype=float
    )

    replacement_probs = (
        replacement_freqs /
        replacement_freqs.sum()
    )

    replacement_char = np.random.choice(
        replacement_chars,
        p=replacement_probs
    )

    perturbed_word = (
        word[:idx]
        + replacement_char
        + word[idx+1:]
    )

    return {
        "position": idx,
        "original_char": original_char,
        "replacement_char": replacement_char,
        "original_word": word,
        "perturbed_word": perturbed_word
    }

In [21]:
MIN_PERTURBED_FREQ = 50

In [22]:
word_freq_lookup = dict(
    zip(word_df["word"], word_df["freq"])
)

In [23]:
from tqdm import tqdm

perturbations = []

for _, row in tqdm(selected_word_df.iterrows(),
                   total=len(selected_word_df)):

    word = row["word"]

    result = perturb_word(word)

    if result is None:
        continue

    perturbed_word = result["perturbed_word"]

    # Skip if perturbation is itself a frequent word
    # (e.g. 再见 -> 在见 should be discarded if 在见 is common enough)
    if word_freq_lookup.get(perturbed_word, 0) >= MIN_PERTURBED_FREQ:
        continue

    result["word_freq"] = row["freq"]
    result["perturbed_freq"] = word_freq_lookup.get(
        perturbed_word,
        0
    )

    perturbations.append(result)

perturbation_df = pd.DataFrame(perturbations)

100%|██████████| 2673/2673 [00:00<00:00, 9109.35it/s]


In [24]:
perturbation_df = pd.DataFrame(
    perturbations
)

perturbation_df.head()

,position,original_char,replacement_char,original_word,perturbed_word,word_freq,perturbed_freq
0,0,中,忠,中国,忠国,20800,0
1,1,个,各,一个,一各,14404,0
2,0,美,每,美国,每国,11073,0
3,0,自,字,自己,字己,8939,0
4,0,可,渴,可能,渴能,8367,0


In [25]:
perturbation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2598 entries, 0 to 2597
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   position          2598 non-null   int64 
 1   original_char     2598 non-null   object
 2   replacement_char  2598 non-null   object
 3   original_word     2598 non-null   object
 4   perturbed_word    2598 non-null   object
 5   word_freq         2598 non-null   int64 
 6   perturbed_freq    2598 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 142.2+ KB


In [26]:
perturbation_df.to_csv("exact_homophone_phrases.csv", index=False)